In [23]:
from pathlib import Path # Import Path so we can work with file/folder paths easily
import pandas as pd
from datetime import datetime

In [24]:
# --- Configuration ---
DATA_FILE   = Path('team_1.parquet')
OUTPUT_DIR  = Path('split_by_year (cleaned and transformed files)')
FILE_PREFIX = 'team_1'

# --- Load ---
print(f"Loading {DATA_FILE}...")
data = pd.read_parquet(DATA_FILE, engine='pyarrow')
print(f"Loaded {len(data):,} rows, {data.shape[1]} columns")

Loading team_1.parquet...
Loaded 3,495,959 rows, 22 columns


In [25]:
# --- TRANSFORM ---

# Convert timestamp
data['timestamp'] = pd.to_datetime(data['timestamp'], utc=True)

# Extract date parts for partitioning
data['year']      = data['timestamp'].dt.year
data['month']     = data['timestamp'].dt.month
data['day']       = data['timestamp'].dt.day

# Extract hour for time of day pollution patterns
data['hour']      = data['timestamp'].dt.hour

# Day of week — 0=Monday, 6=Sunday (Weekday vs weekend pollution patterns)
data['day_of_week'] = data['timestamp'].dt.dayofweek

# Season based on month (Seasonal trends in weather and pollution)
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'

data['season'] = data['month'].apply(get_season)

# Time of day category (Morning rush vs night — pollution spikes by time)
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour < 21:
        return 'evening'
    else:
        return 'night'

data['time_of_day'] = data['hour'].apply(get_time_of_day)

# Normalise pollutant value between 0 and 1 per pollutant type (Compare pollutants on same scale )
data['value_normalised'] = data.groupby('pollutant')['value'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

print("Transformation complete")
print(data[['timestamp', 'year', 'month', 'day', 'hour', 
            'day_of_week', 'season', 'time_of_day', 'value_normalised']].head())

Transformation complete
                  timestamp  year  month  day  hour  day_of_week  season  \
0 2024-01-01 00:00:00+00:00  2024      1    1     0            0  winter   
1 2024-01-01 00:00:00+00:00  2024      1    1     0            0  winter   
2 2024-01-01 00:00:00+00:00  2024      1    1     0            0  winter   
3 2024-01-01 00:00:00+00:00  2024      1    1     0            0  winter   
4 2024-01-01 00:00:00+00:00  2024      1    1     0            0  winter   

  time_of_day  value_normalised  
0       night          0.004360  
1       night          0.091892  
2       night          0.018011  
3       night          0.053459  
4       night          0.031419  


In [26]:
# --- DATA CLEANING ---

print(f"Shape before cleaning: {data.shape}")

# --- Drop completely empty columns ---
empty_cols = [col for col in data.columns if data[col].isna().all()]
print(f"Dropping empty columns: {empty_cols}")
data = data.drop(columns=empty_cols)

# --- Drop redundant columns ---
data = data.drop(columns=['datetime'], errors='ignore')
print("Dropped redundant datetime column")

# --- Drop fully duplicate rows ---
before = len(data)
data = data.drop_duplicates()
print(f"Dropped {before - len(data):,} duplicate rows")

# --- Drop rows where critical columns are null ---
critical_cols = ['station_id', 'timestamp', 'pollutant', 'value']
before = len(data)
data = data.dropna(subset=critical_cols)
print(f"Dropped {before - len(data):,} rows with nulls in critical columns")

# --- Fill small gaps with forward fill (for stations with mostly good data) ---
weather_cols = ['at_c', 'rh_percent', 'ws_m_s', 'wd_deg', 'rf_mm', 'sr_w_mt2', 'bp_mmhg']
data = data.sort_values(['station_id', 'timestamp'])
data[weather_cols] = data.groupby('station_id')[weather_cols].ffill()
data[weather_cols] = data.groupby('station_id')[weather_cols].bfill()

# --- For stations with no weather data at all, fill with overall mean ---
# This handles site_104 and site_108
for col in weather_cols:
    overall_mean = data[col].mean()
    data[col] = data[col].fillna(overall_mean)


# Any remaining nulls at the start of a station's records get backfilled
data[weather_cols] = data.groupby('station_id')[weather_cols].bfill()

print(f"Forward filled missing weather readings")

# --- Confirm nulls remaining ---
nulls_remaining = data.isnull().sum()
nulls_remaining = nulls_remaining[nulls_remaining > 0]
if nulls_remaining.empty:
    print("No nulls remaining")
else:
    print(f"Nulls remaining:\n{nulls_remaining}")

print(f"Shape after cleaning: {data.shape}")

Shape before cleaning: (3495959, 26)
Dropping empty columns: ['vws_m_s']
Dropped redundant datetime column
Dropped 0 duplicate rows
Dropped 0 rows with nulls in critical columns
Forward filled missing weather readings
No nulls remaining
Shape after cleaning: (3495959, 24)


In [27]:

# --- Partition and Save ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for (year, month, day), df_day in data.groupby(['year', 'month', 'day']):
    day_dir = OUTPUT_DIR / str(year) / f'{month:02d}'
    day_dir.mkdir(parents=True, exist_ok=True)

    output_path = day_dir / f'{FILE_PREFIX}_{year}_{month:02d}_{day:02d}.csv'
    df_day.drop(columns=['year', 'month', 'day']).to_csv(output_path, index=False)
    print(f"Saved {len(df_day):,} rows → {output_path}")

print(f"\nDone! Files saved to {OUTPUT_DIR.resolve()}")

Saved 4,729 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_01.csv
Saved 4,793 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_02.csv
Saved 4,859 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_03.csv
Saved 5,094 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_04.csv
Saved 5,097 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_05.csv
Saved 5,049 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_06.csv
Saved 5,082 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_07.csv
Saved 5,083 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_08.csv
Saved 5,074 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_09.csv
Saved 5,077 rows → split_by_year (cleaned and transformed files)\2024\01\team_1_2024_01_10.csv
Saved 5,067 rows → split_by_year (cleaned and tran

In [31]:
df24 = pd.read_csv('split_by_year (cleaned and transformed files)/2024/01/team_1_2024_01_01.csv')
df24.head()

,station_id,state,city,station_name,timestamp,at_c,rh_percent,ws_m_s,wd_deg,rf_mm,...,sr_w_mt2,bp_mmhg,pollutant,value,station,hour,day_of_week,season,time_of_day,value_normalised
0,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,co,2.26,"Burari Crossing, Delhi - IMD",0,0,winter,night,0.045227
1,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,no,11.67,"Burari Crossing, Delhi - IMD",0,0,winter,night,0.023330
2,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,no2,5.14,"Burari Crossing, Delhi - IMD",0,0,winter,night,0.010273
3,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,ozone,19.94,"Burari Crossing, Delhi - IMD",0,0,winter,night,0.040092
4,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2024-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,pm10,201.09,"Burari Crossing, Delhi - IMD",0,0,winter,night,0.201265


In [32]:
df25 = pd.read_csv('split_by_year (cleaned and transformed files)/2025/01/team_1_2025_01_01.csv')
df25.head()

,station_id,state,city,station_name,timestamp,at_c,rh_percent,ws_m_s,wd_deg,rf_mm,...,sr_w_mt2,bp_mmhg,pollutant,value,station,hour,day_of_week,season,time_of_day,value_normalised
0,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2025-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,pm10,270.90,"Burari Crossing, Delhi - IMD",0,2,winter,night,0.271139
1,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2025-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,co,0.92,"Burari Crossing, Delhi - IMD",0,2,winter,night,0.018411
2,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2025-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,no,24.02,"Burari Crossing, Delhi - IMD",0,2,winter,night,0.048040
3,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2025-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,no2,11.41,"Burari Crossing, Delhi - IMD",0,2,winter,night,0.022828
4,site_104,Delhi,Delhi,"Burari Crossing, Delhi - IMD",2025-01-01 00:00:00+00:00,25.761743,63.614581,1.286361,150.758626,0.033738,...,125.007555,930.175799,ozone,6.15,"Burari Crossing, Delhi - IMD",0,2,winter,night,0.012351


In [35]:
# Sample 10% of your data — enough for meaningful benchmarks
sample = data.sample(frac=0.1, random_state=42)
print(f"Sample size: {len(sample):,} rows")

# Build models from sample instead of full data
stations_df   = sample[['station_id', 'state', 'city', 'station_name']].drop_duplicates()
readings_df   = sample[['station_id', 'timestamp', 'at_c', 'rh_percent', 'ws_m_s', 'rf_mm', 'bp_mmhg']]
pollutants_df = sample[['station_id', 'timestamp', 'pollutant', 'value']]

# NoSQL from sample
nosql_docs = sample.to_dict(orient='records')

# Wide table from sample
wide_df = sample.pivot_table(
    index=['station_id', 'state', 'city', 'station_name', 'timestamp'],
    columns='pollutant',
    values='value',
    aggfunc='mean'
).reset_index()

wide_df.columns.name = None
wide_df = wide_df.merge(
    sample[['station_id', 'timestamp', 'at_c', 'rh_percent', 'ws_m_s', 'rf_mm', 'bp_mmhg']].drop_duplicates(),
    on=['station_id', 'timestamp']
)

Sample size: 349,596 rows


In [36]:
import time

def benchmark(label, func):
    start = time.time()
    result = func()
    end = time.time()
    duration = (end - start) * 1000
    count = len(result) if hasattr(result, '__len__') else 'N/A'
    print(f"{label:<45} {duration:>8.2f}ms   {count:>10} rows")
    return result

In [37]:
print(f"\n{'Query':<45} {'Time':>8}     {'Rows':>10}")
print("=" * 70)

# ── Query 1: Filter by city ──────────────────────────────────────────
city = 'Delhi'

benchmark(f"[RDBMS]  Filter city = {city}",
    lambda: readings_df[readings_df['station_id'].isin(
        stations_df[stations_df['city'] == city]['station_id']
    )])

benchmark(f"[NoSQL]  Filter city = {city}",
    lambda: [d for d in nosql_docs if d.get('city') == city])

benchmark(f"[Wide]   Filter city = {city}",
    lambda: wide_df[wide_df['city'] == city])

print()

# ── Query 2: Filter by month ─────────────────────────────────────────
benchmark("[RDBMS]  Filter month = 6",
    lambda: readings_df[pd.to_datetime(readings_df['timestamp']).dt.month == 6])

benchmark("[NoSQL]  Filter month = 6",
    lambda: [d for d in nosql_docs if pd.to_datetime(d.get('timestamp')).month == 6])

benchmark("[Wide]   Filter month = 6",
    lambda: wide_df[pd.to_datetime(wide_df['timestamp']).dt.month == 6])

print()

# ── Query 3: Average temperature per city ────────────────────────────
benchmark("[RDBMS]  Avg temp per city (with join)",
    lambda: readings_df.merge(stations_df, on='station_id').groupby('city')['at_c'].mean())

benchmark("[NoSQL]  Avg temp per city",
    lambda: pd.DataFrame(nosql_docs).groupby('city')['at_c'].mean())

benchmark("[Wide]   Avg temp per city",
    lambda: wide_df.groupby('city')['at_c'].mean())

print()

# ── Query 4: Pollutant lookup ─────────────────────────────────────────
benchmark("[RDBMS]  pm25 readings (filter pollutant col)",
    lambda: pollutants_df[pollutants_df['pollutant'] == 'pm25'])

benchmark("[NoSQL]  pm25 readings",
    lambda: [d for d in nosql_docs if d.get('pollutant') == 'pm25'])

benchmark("[Wide]   pm25 readings",
    lambda: wide_df[['station_id', 'city', 'timestamp', 'pm25']].dropna())


Query                                             Time           Rows
[RDBMS]  Filter city = Delhi                     40.88ms       349596 rows
[NoSQL]  Filter city = Delhi                     80.35ms       349596 rows
[Wide]   Filter city = Delhi                     73.03ms       238704 rows

[RDBMS]  Filter month = 6                        63.54ms        28018 rows
[NoSQL]  Filter month = 6                       290.69ms        28018 rows
[Wide]   Filter month = 6                        26.68ms        19214 rows

[RDBMS]  Avg temp per city (with join)           94.95ms            1 rows
[NoSQL]  Avg temp per city                     1829.63ms            1 rows
[Wide]   Avg temp per city                       18.09ms            1 rows

[RDBMS]  pm25 readings (filter pollutant col)    32.27ms        39484 rows
[NoSQL]  pm25 readings                           69.02ms        39484 rows
[Wide]   pm25 readings                           41.20ms        39484 rows


,station_id,city,timestamp,pm25
0,site_104,Delhi,2024-01-01 00:30:00+00:00,114.27
1,site_104,Delhi,2024-01-01 01:00:00+00:00,106.18
7,site_104,Delhi,2024-01-01 03:15:00+00:00,97.11
11,site_104,Delhi,2024-01-01 06:30:00+00:00,194.84
17,site_104,Delhi,2024-01-01 10:15:00+00:00,96.17
...,...,...,...,...
238664,site_5024,Delhi,2025-12-31 08:30:00+00:00,127.00
238666,site_5024,Delhi,2025-12-31 09:15:00+00:00,146.00
238669,site_5024,Delhi,2025-12-31 11:45:00+00:00,146.00
238695,site_5024,Delhi,2025-12-31 20:30:00+00:00,355.00


In [12]:
!pip install psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   -------------------------------

In [38]:
from sqlalchemy import create_engine, text

USERNAME = "postgres"
PASSWORD = "admin123"
HOST     = "localhost"
PORT     = "5434"
DATABASE = "DigitalTransformation"

engine = create_engine(
    f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

print("Database connection successful")

Database connection successful


In [42]:
# =============================================================
# SERVE TO RDBMS (PostgreSQL)
# =============================================================

# --- Stations table ---
stations_df = data[['station_id', 'state', 'city', 'station_name']]\
    .drop_duplicates()\
    .reset_index(drop=True)

stations_df.to_sql(
    'stations',
    engine,
    if_exists='replace',
    index=False
)
print(f"Loaded {len(stations_df):,} rows into stations table")

# --- Weather readings table ---
readings_df = data[['station_id', 'timestamp', 'at_c', 'rh_percent',
                     'ws_m_s', 'wd_deg', 'rf_mm', 'sr_w_mt2', 'bp_mmhg']]\
    .drop_duplicates()\
    .reset_index(drop=True)

readings_df.to_sql(
    'weather_readings',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10000
)
print(f"Loaded {len(readings_df):,} rows into weather_readings table")

# --- Pollutant readings table ---
pollutants_df = data[['station_id', 'timestamp', 'pollutant', 'value']]\
    .reset_index(drop=True)

pollutants_df.to_sql(
    'pollutant_readings',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10000
)
print(f"Loaded {len(pollutants_df):,} rows into pollutant_readings table")

# =============================================================
# SERVE TO WIDE TABLE (partitioned by month)
# =============================================================

wide_df = data.pivot_table(
    index=['station_id', 'state', 'city', 'station_name', 'timestamp',
           'year', 'month', 'day', 'hour', 'at_c', 'rh_percent',
           'ws_m_s', 'wd_deg', 'rf_mm', 'sr_w_mt2', 'bp_mmhg'],
    columns='pollutant',
    values='value',
    aggfunc='mean'
).reset_index()

wide_df.columns.name = None

wide_dir = Path('wide_table')
wide_dir.mkdir(exist_ok=True)

for (year, month), df_month in wide_df.groupby(['year', 'month']):
    path = wide_dir / f'wide_{year}_{month:02d}.csv'
    df_month.to_csv(path, index=False)
    print(f"Saved wide table {year}-{month:02d} → {path}")

print(f"\nDone! Wide table saved to {wide_dir.resolve()}")

# =============================================================
# VERIFY POSTGRESQL TABLES
# =============================================================

with engine.connect() as conn:
    for table in ['stations', 'weather_readings', 'pollutant_readings']:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"{table}: {count:,} rows")

Loaded 6 rows into stations table
Loaded 409,427 rows into weather_readings table
Loaded 3,495,959 rows into pollutant_readings table
Saved wide table 2024-01 → wide_table\wide_2024_01.csv
Saved wide table 2024-02 → wide_table\wide_2024_02.csv
Saved wide table 2024-03 → wide_table\wide_2024_03.csv
Saved wide table 2024-04 → wide_table\wide_2024_04.csv
Saved wide table 2024-05 → wide_table\wide_2024_05.csv
Saved wide table 2024-06 → wide_table\wide_2024_06.csv
Saved wide table 2024-07 → wide_table\wide_2024_07.csv
Saved wide table 2024-08 → wide_table\wide_2024_08.csv
Saved wide table 2024-09 → wide_table\wide_2024_09.csv
Saved wide table 2024-10 → wide_table\wide_2024_10.csv
Saved wide table 2024-11 → wide_table\wide_2024_11.csv
Saved wide table 2024-12 → wide_table\wide_2024_12.csv
Saved wide table 2025-01 → wide_table\wide_2025_01.csv
Saved wide table 2025-02 → wide_table\wide_2025_02.csv
Saved wide table 2025-03 → wide_table\wide_2025_03.csv
Saved wide table 2025-04 → wide_table\wid

In [43]:
!pip install eralchemy2

  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build pygraphviz


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [94 lines of output]
  C:\Users\Administrator\AppData\Local\Temp\pip-build-env-2nwqfgjp\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
  !!
  
          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
  
          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
          ********************************************************************************
  
  !!
    corresp(dist, value, root_dir)
  C:\Users\Administrator\AppData\Local

In [ ]:
from eralchemy2 import render_er

render_er(
    f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}",
    'er_diagram.png'
)

# Display it in the notebook
from IPython.display import Image
Image('er_diagram.png')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# --- Stations table ---
stations_box = mpatches.FancyBboxPatch((0.5, 5), 2.5, 2.5,
    boxstyle="round,pad=0.1", linewidth=2,
    edgecolor='#2c3e50', facecolor='#3498db', alpha=0.8)
ax.add_patch(stations_box)
ax.text(1.75, 7.2, 'STATIONS', ha='center', va='center',
    fontsize=11, fontweight='bold', color='white')
ax.text(1.75, 6.8, '─────────────', ha='center', color='white', fontsize=8)
ax.text(1.75, 6.5, '🔑 station_id (PK)', ha='center', color='white', fontsize=8)
ax.text(1.75, 6.2, 'state', ha='center', color='white', fontsize=8)
ax.text(1.75, 5.9, 'city', ha='center', color='white', fontsize=8)
ax.text(1.75, 5.6, 'station_name', ha='center', color='white', fontsize=8)

# --- Weather readings table ---
weather_box = mpatches.FancyBboxPatch((3.8, 4.5), 2.5, 3,
    boxstyle="round,pad=0.1", linewidth=2,
    edgecolor='#2c3e50', facecolor='#27ae60', alpha=0.8)
ax.add_patch(weather_box)
ax.text(5.05, 7.2, 'WEATHER_READINGS', ha='center', va='center',
    fontsize=11, fontweight='bold', color='white')
ax.text(5.05, 6.8, '─────────────', ha='center', color='white', fontsize=8)
ax.text(5.05, 6.5, '🔑 station_id (FK)', ha='center', color='white', fontsize=8)
ax.text(5.05, 6.2, 'timestamp', ha='center', color='white', fontsize=8)
ax.text(5.05, 5.9, 'at_c', ha='center', color='white', fontsize=8)
ax.text(5.05, 5.6, 'rh_percent', ha='center', color='white', fontsize=8)
ax.text(5.05, 5.3, 'ws_m_s', ha='center', color='white', fontsize=8)
ax.text(5.05, 5.0, 'rf_mm / bp_mmhg', ha='center', color='white', fontsize=8)
ax.text(5.05, 4.7, 'sr_w_mt2 / wd_deg', ha='center', color='white', fontsize=8)

# --- Pollutant readings table ---
pollutant_box = mpatches.FancyBboxPatch((3.8, 1), 2.5, 3,
    boxstyle="round,pad=0.1", linewidth=2,
    edgecolor='#2c3e50', facecolor='#e74c3c', alpha=0.8)
ax.add_patch(pollutant_box)
ax.text(5.05, 3.7, 'POLLUTANT_READINGS', ha='center', va='center',
    fontsize=11, fontweight='bold', color='white')
ax.text(5.05, 3.3, '─────────────', ha='center', color='white', fontsize=8)
ax.text(5.05, 3.0, '🔑 station_id (FK)', ha='center', color='white', fontsize=8)
ax.text(5.05, 2.7, 'timestamp', ha='center', color='white', fontsize=8)
ax.text(5.05, 2.4, 'pollutant', ha='center', color='white', fontsize=8)
ax.text(5.05, 2.1, 'value', ha='center', color='white', fontsize=8)
ax.text(5.05, 1.8, 'year / month / day', ha='center', color='white', fontsize=8)
ax.text(5.05, 1.5, 'hour / season', ha='center', color='white', fontsize=8)

# --- Wide table ---
wide_box = mpatches.FancyBboxPatch((7.2, 2.5), 2.5, 3,
    boxstyle="round,pad=0.1", linewidth=2,
    edgecolor='#2c3e50', facecolor='#9b59b6', alpha=0.8)
ax.add_patch(wide_box)
ax.text(8.45, 5.2, 'WIDE_TABLE', ha='center', va='center',
    fontsize=11, fontweight='bold', color='white')
ax.text(8.45, 4.8, '─────────────', ha='center', color='white', fontsize=8)
ax.text(8.45, 4.5, 'station_id', ha='center', color='white', fontsize=8)
ax.text(8.45, 4.2, 'timestamp', ha='center', color='white', fontsize=8)
ax.text(8.45, 3.9, 'at_c / rh_percent', ha='center', color='white', fontsize=8)
ax.text(8.45, 3.6, 'pm25 / pm10 / no2', ha='center', color='white', fontsize=8)
ax.text(8.45, 3.3, 'co / so2 / benzene', ha='center', color='white', fontsize=8)
ax.text(8.45, 3.0, 'season / time_of_day', ha='center', color='white', fontsize=8)
ax.text(8.45, 2.7, '[partitioned by month]', ha='center',
    color='#f0f0f0', fontsize=7, style='italic')

# --- Relationship lines ---
# Stations → Weather readings
ax.annotate('', xy=(3.8, 6.2), xytext=(3.0, 6.2),
    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))
ax.text(3.4, 6.35, '1:N', ha='center', fontsize=8, color='#2c3e50', fontweight='bold')

# Stations → Pollutant readings
ax.annotate('', xy=(3.8, 2.5), xytext=(1.75, 5.0),
    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))
ax.text(2.5, 3.6, '1:N', ha='center', fontsize=8, color='#2c3e50', fontweight='bold')

# RDBMS → Wide table arrow
ax.annotate('', xy=(7.2, 4.0), xytext=(6.3, 4.0),
    arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=1.5, linestyle='dashed'))
ax.text(6.75, 4.2, 'pivot', ha='center', fontsize=8,
    color='#7f8c8d', style='italic')

# --- Title ---
ax.text(5.0, 7.7, 'DigiTransformation Project — Data Model',
    ha='center', fontsize=13, fontweight='bold', color='#2c3e50')

plt.tight_layout()
plt.savefig('er_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print("ER diagram saved as er_diagram.png")